# Confidence-gated memory guidance 

An OCR model reads every invoice as if it had never seen one before. Vendors print the same layout over and over, so a **memory of where each field sits** should help. But a memory is only useful if it knows when a remembered position from the wrong vendor, or from a layout the vendor has since changed, points the model at the wrong spot.

This demo walks through the design: a **rule-group memory graph** in Neo4j, and a **gate** that shows the model a
remembered position only when it can say how likely that position is to be right.

It is the rule-group version of `demo.ipynb`. Every invoice page is stored as a **path** of the entities and tables
it has, and pages that agree on structure and position form a **rule group**. Parts II–IV ask the same questions
of the new memory.

| Part | Question |
|---|---|
| **I. The memory graph** | What is remembered, and how do raw observations become rules? |
| **II. Confidence gating** | Which rule group is this page, and how far can each rule be trusted? |
| **III. Evaluation, no LLM** | What can the gate do on generated test pages: look-alike vendors, missing vendor names, unseen vendors, redesigned layouts? |
| **IV. Does it help the LLM?** | A short summary of the Gemini runs in `gating.ipynb` |

### Setup
Settings live in `src/config.py` (the same values as `gating.ipynb`). Override them here. `data_dir` must hold
`input/` (PDFs) and `output/` (ground-truth JSON). The rule-group memory runs in its **own** Neo4j container, so it never overwrites the graph `demo.ipynb` builds:
```
docker run --name graph_gating_rules -d -p 7476:7474 -p 7689:7687 -e NEO4J_AUTH=neo4j/gating-local neo4j:5.26-community
```

In [ ]:
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

import helpers as H
from src import viz
from src import evaluation as E
from src.config import Config
from src.data import NEW, Corpus
from src.gate import Gate
from src.memory_graph import BOX_KEYS, FEATURES
from src.posterior import LayoutPosterior
from src.rule_memory import UNIDENTIFIED, RuleMemory
from src.scenarios import ScenarioGenerator, case_table, describe_noise, entity_agreement, find_twins

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

cfg = Config(
    data_dir=Path.cwd().parent / "data",     # <- point at the folder holding input/ and output/
    neo4j_uri="bolt://localhost:7689", neo4j_user="neo4j", neo4j_password="gating-local",
)
REBUILD_GRAPH = True                         # False keeps an already-built graph for the same split

In [ ]:
corpus = Corpus.load(cfg)
graph = RuleMemory(cfg)
graph.build(corpus, reset=REBUILD_GRAPH)
post = LayoutPosterior(graph, corpus, cfg, p_new=graph.p_new)
gate = Gate(post, graph)

vendors = corpus.vendor_table()
print(f"{len(corpus.records)} invoices from {len(vendors)} vendors | memory built from {len(corpus.train_ids)} "
      f"| held out {len(corpus.test_ids)}")
corpus.bucket_table()

The split is **per vendor**: two thirds of each vendor's invoices build the memory, and the rest are held out. A
vendor with only one invoice lands entirely on one side.

In [ ]:
EX_VENDOR = vendors[vendors.n_test > 0].sort_values("n_train", ascending=False).vendor_key.iloc[0]
EX_DOC = sorted(d for d in corpus.test_ids if corpus.vendor_of(d) == EX_VENDOR)[0]
EX_PAGE = corpus.records[EX_DOC]["pages"][0]
EX_LAYOUT = graph.cypher("MATCH (:Vendor {key: $v})-[:HAS_RULE_GROUP]->(g:RuleGroup {page: 1}) "
                         "RETURN g.id AS id ORDER BY g.n_pages DESC, g.id LIMIT 1", v=EX_VENDOR)[0]["id"]
print(f"Running example: {EX_DOC}, page 1, vendor '{EX_VENDOR}' "
      f"({corpus.train_count[EX_VENDOR]} invoices in memory) -> rule group {EX_LAYOUT}")

---
## Part I: The memory graph

### 1. The schema: a page is a path, a rule group is an agreed path
A document page becomes a **path** of only what it has: one `Field` node per filled entity and one `Table` node per
table, in schema order, each carrying its box. A **rule group** is the same kind of path, rooted at the group: each
`EntityRule` or `TableRule` is an entity that earlier, similar pages agreed should be printed in a given region,
with the share of the group's pages that have it (`fill_rate`). Rule groups sit under their vendor, and every group
records the `document_type` it guides (here always `Invoice`).

```
(:Vendor {key, names})─HAS_RULE_GROUP─►(:RuleGroup {id, document_type, page, n_pages, structure_vector})
                                              └──►(:EntityRule {range, reliability, fill_rate})──► … ──►(:TableRule)
(:Vendor)─HAS_DOCUMENT─►(:Document {doc_id, document_type})─HAS_PAGE─►(:Page {number})
                                              └──►(:Field {key, bbox, label})──► … ──►(:Table {type, region, rows})
(:Page)─IN_GROUP {role: founded | joined}─►(:RuleGroup)
(:Field)─SUPPORTS─►(:EntityRule)                      (:Table)─SUPPORTS─►(:TableRule)
(:EntityRule|Field)─OF_KIND─►(:Entity {key})─NORMALIZED_BY─►(:NormalizationRule)
                                               └─CONSTRAINED_BY─►(:Invariant)
```
The unnamed arrows only chain a path together. The named edges carry meaning: which group a page belongs to,
and which observation supports which rule. A missing field is a **missing node**, so two groups with different
fields have visibly different paths.

Below is a small rule group with a table (orange root), the page that founded it (grey root), and the `SUPPORTS`
edges between the page's nodes and the group's rules: two parallel paths joined rung by rung. Two entities are
followed to the shared dictionary nodes.

In [ ]:
# the smallest multi-page group whose path ends in a table: short enough to read every caption
small = graph.groups_table().query("pages >= 2 and nodes >= 5 and signature.str.contains('Table')", engine="python")
SCHEMA_GROUP = small.sort_values(["nodes", "pages"], ascending=[True, False]).group.iloc[0]
SCHEMA_PAGE = graph.cypher("MATCH (p:Page)-[:IN_GROUP {role: 'founded'}]->(:RuleGroup {id: $g}) RETURN p.id AS id",
                           g=SCHEMA_GROUP)[0]["id"]
print(f"Rule group {SCHEMA_GROUP}, founded by page {SCHEMA_PAGE}")
viz.show_graph(graph.driver, '''
    MATCH (v:Vendor)-[h:HAS_RULE_GROUP]->(g:RuleGroup {id: $gid})
    MATCH rule_path = (g)-[:`_`*]->(r)
    MATCH doc = (v)-[:HAS_DOCUMENT]->(:Document)-[:HAS_PAGE]->(p:Page {id: $pid})
    MATCH page_path = (p)-[:`_`*]->(f)-[s:SUPPORTS]->(r)
    MATCH (p)-[m:IN_GROUP]->(g)
    OPTIONAL MATCH (r)-[o:OF_KIND]->(e:Entity) WHERE e.key IN ['Invoice Amount', 'Invoice Date']
    OPTIONAL MATCH (e)-[n:NORMALIZED_BY]->(nr:NormalizationRule)
    OPTIONAL MATCH (e)-[c:CONSTRAINED_BY]->(i:Invariant)
    RETURN v, h, rule_path, doc, page_path, s, m, o, e, n, nr, c, i''', gid=SCHEMA_GROUP, pid=SCHEMA_PAGE,
    height="640px")

In [ ]:
graph.node_counts()

### 2. Writing memory: ingestion
Training pages are ingested one at a time, in `doc_id` order. For each page:

1. **Vendor check.** The vendor name maps to a vendor key. An unseen key creates a new `Vendor`; a page with no
   vendor name goes under the general **unidentified** vendor.
2. **The page becomes a path** under its document (`Field` and `Table` nodes, with their boxes).
3. **Similar candidates.** The page and each of the vendor's rule groups (same page type) become **vectors**: which
   entities and tables are present, and roughly where (soft 4 × 4 position bins). A group's vector is the mean of
   its pages'. Groups with cosine ≥ `rule_group_similarity` (0.85) are candidates, so a page that only adds or
   drops an optional field can still match.
4. **Range check.** For each candidate, every node the page and the group **share** must fall inside the group's
   range for it (an entity's box centre; a table's top-left corner, since a table grows down with its rows).
5. **Join or found.** If a candidate passes, the page **joins** the most similar one: it gets an `IN_GROUP` edge,
   its nodes get `SUPPORTS` edges, the group's path grows by any node it lacked, and its rules are recomputed.
   Otherwise the page **founds** a new rule group whose path copies its own.

The log below is where every training page of the running example's vendor went. `candidates` counts groups that
were similar enough, `passed` those whose ranges held the page, and `similarity` is the cosine with the group it
joined. A page that founded a group despite having candidates was split off by the range check: similar fields,
printed elsewhere.

In [ ]:
log = graph.ingestion_log()
display(graph.ingestion_log(EX_VENDOR).drop(columns="vendor"))
founded = log[log.role == "founded"]
print(f"All {len(log)} training pages: {(log.role == 'joined').sum()} joined a group, {len(founded)} founded one "
      f"({(founded.candidates == 0).sum()} had no similar group, {(founded.candidates > 0).sum()} "
      f"failed the range check).")
viz.show_graph(graph.driver, '''
    MATCH (v:Vendor {key: $vendor})-[h:HAS_RULE_GROUP]->(g:RuleGroup {page: 1})<-[m:IN_GROUP]-(p:Page)
    RETURN v, h, g, m, p''', vendor=EX_VENDOR,
    caption=lambda n: n.properties["id"][:8] if n.properties["labels"][0] == "Page" else viz.default_caption(n),
    rel_caption=lambda r: r.properties.get("role", ""))

### 3. One vendor, several rule groups
A vendor doesn't always print the same thing. A page whose fields or positions differ enough has a cosine below
0.85 with the vendor's groups. A page with similar fields printed **elsewhere** fails the range check. Either way
it founds its own group, with its own rules. Because a group's path holds only the nodes its pages had, the
difference is visible in the graph itself. Below is the vendor's largest page-1 group next to the groups that
differ from it most; the nodes that differ are drawn large and named.

Ingestion is greedy and in order: a page is checked against the groups as they stand **when it arrives**. A page
split off early, while the large group still had one page and the narrowest ranges, can sit beside a group it
would join today.

In [ ]:
groups = graph.groups_table()
multi = (groups[groups.vendor != UNIDENTIFIED].groupby(["vendor", "page"])
         .agg(groups=("group", "size"), pages=("pages", list)).query("groups >= 2")
         .sort_values(["groups", "vendor"], ascending=[False, True]))
display(multi.head(8))
VAR_VENDOR = EX_VENDOR if (EX_VENDOR, 1) in multi.index else multi.index[0][0]
own = groups[(groups.vendor == VAR_VENDOR) & (groups.page == 1)]
paths = groups.set_index("group").signature.str.split(" > ")
base = post.vendor_layouts[own.group.iloc[0]]


def status(group_id, key):   # how a rule of another group compares with the vendor's largest group
    if group_id == base["id"]:
        return "largest group"
    if key not in paths[base["id"]]:
        return "not in the largest group"
    other = post.vendor_layouts[group_id]["rules"].get(key)
    if other and key in base["rules"] and not H.range_overlap(other["range"], base["rules"][key]["range"]):
        return "printed elsewhere"
    return "same as the largest group"


STATUS_COLORS = {"not in the largest group": viz.MAGENTA, "printed elsewhere": viz.RED,
                 "same as the largest group": viz.GRID}
nodes = {(g, k): status(g, k) for g in own.group for k in paths[g]}
table = pd.DataFrame([{"group": g, "pages": p, "nodes": n, "same as the largest group":
                       sum(nodes[(g, k)] == "same as the largest group" for k in paths[g]),
                       **{s: [k for k in paths[g] if nodes[(g, k)] == s] for s in list(STATUS_COLORS)[:2]}}
                      for g, p, n in own[["group", "pages", "nodes"]].itertuples(index=False)])
display(table)
# the largest group and the two that differ from it most
differ = table.iloc[1:].assign(n=lambda d: d["not in the largest group"].map(len) + d["printed elsewhere"].map(len))
VAR_GROUPS = [base["id"]] + differ.sort_values(["n", "pages"], ascending=False).group.head(2).tolist()


def is_rule(n):
    return n.properties["labels"][0] in ("EntityRule", "TableRule")


def node_status(n):
    g = n.properties["id"].rsplit("|", 1)[0]
    return "largest group" if g == base["id"] else nodes[(g, n.properties["key"])]


def differs(n):
    return is_rule(n) and node_status(n) in ("not in the largest group", "printed elsewhere")


viz.show_graph(graph.driver, '''
    MATCH (v:Vendor {key: $vendor})-[h:HAS_RULE_GROUP]->(g:RuleGroup) WHERE g.id IN $groups
    MATCH path = (g)-[:`_`*]->()
    RETURN v, h, path''', vendor=VAR_VENDOR, groups=VAR_GROUPS, height="640px",
    color=lambda n: (viz.YELLOW if node_status(n) == "largest group" else STATUS_COLORS[node_status(n)]) if is_rule(n)
                    else viz.LABEL_COLORS.get(n.properties["labels"][0], viz.MUTED),
    size=lambda n: 48 if differs(n) or not is_rule(n) else 20,
    caption=lambda n: viz.default_caption(n) if differs(n) or not is_rule(n) else "",
    legend={"vendor": viz.BLUE, "rule group": viz.ORANGE, "rule of the largest group": viz.YELLOW,
            **{f"rule {k}": c for k, c in STATUS_COLORS.items()
               if any(v == k for (g, _), v in nodes.items() if g in VAR_GROUPS)}})

### 4. Consolidation: from observations to a rule with a reliability
Every time a page joins a group, each rule node of the group is recomputed from the observations that `SUPPORTS`
it:

1. **Peak.** The observed box centre with the most other centres within `bbox_tolerance` (0.03 of the page).
   **Support** is that count.
2. **Strength.** The rule is *strong* with ≥ 3 agreeing pages, *weak* with 2, and *single* when the group has one
   page.
3. **Range.** A square around the peak, half-width `clip(2 × spread, 0.06, 0.15)`. Extracted boxes are noisy
   (Gemini's centres miss by ~0.057), so a rule must be matched against a region, not a point.
4. **`reliability`.** The chance that the next page of this group prints the value inside the range. A page only
   joins when it is inside the ranges, so counting members would flatter the rule. Instead each observation is
   tested **leave-one-out**, against the range rebuilt from the other observations. With $h$ of $n$ such hits and an
   empirical-Bayes Beta prior, $\text{reliability} = \frac{h + \alpha}{n + \alpha + \beta}$.

The plot shows every observed centre of one rule of the running example's group, its peak, its range and its
typical box. The table on the right is the whole memory: most groups hold a single page, so most rules carry only
the prior's reliability. How far the gate trusts them comes from the posterior in Part II, not from repeated
observations.

In [ ]:
rules_tbl = graph.rules_table()
# the running group's rule whose boxes scatter most: the one where peak, range and leave-one-out all show
cand = rules_tbl[(rules_tbl.group == EX_LAYOUT) & (rules_tbl.n_boxes >= 3)].assign(
    misses=lambda d: d.n_boxes - d.loo_hits).sort_values(["misses", "spread"], ascending=False)
EX_KEY = cand.entity.iloc[0]
o = graph.entity_observations(EX_LAYOUT, EX_KEY)
centres = [tuple(p) for p in o["observations"].point if p is not None]

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 7.5), gridspec_kw={"width_ratios": [1.15, 1]})
viz.page_axes(corpus.pdf(EX_DOC), 1, ax, title=f"'{EX_KEY}' in {EX_LAYOUT}", fade=0.45)
ax.scatter(*zip(*centres), s=28, color=viz.BLUE, edgecolor="white", lw=1, zorder=5, label="observed centres")
ax.scatter(*o["center"], s=90, marker="X", color=viz.ORANGE, edgecolor="white", lw=1, zorder=6, label="peak")
viz.draw_range(ax, o["range"], viz.ORANGE, f"range · reliability = {o['reliability']:.2f}")
viz.draw_box(ax, o["median"], viz.INK, None, lw=1.2)
ax.legend(loc="lower left", fontsize=8, frameon=True)

by_strength = rules_tbl.groupby("strength").reliability.describe()[["count", "mean", "min", "max"]]
ax2.axis("off")
ax2.set_title("All rules: strength label vs reliability", loc="left")
ax2.table(cellText=by_strength.round(2).values, rowLabels=by_strength.index, colLabels=by_strength.columns,
          loc="upper left", bbox=[0.1, 0.62, 0.9, 0.3])
ax2.text(0, 0.45, f"support {o['support']} of {o['n_boxes']} boxes\n"
                  f"{o['loo_hits']} of {o['n_boxes']} inside the range rebuilt without them\n"
                  f"strength '{o['strength']}' · reliability {o['reliability']:.3f}\n\n"
                  f"Empirical prior: one page's box lands near another page's\nbox of the same structure "
                  f"{graph.rule_prior[2]:.0%} of the time -> Beta({graph.rule_prior[0]:.2f}, {graph.rule_prior[1]:.2f})",
         va="top", fontsize=9, color=viz.INK_2)
plt.show()

### 5. What is true of invoices in general
Two parts of the memory don't belong to one known vendor:

* **The unidentified vendor.** Pages printed without a vendor name are grouped under it, by structure and position
  alone.
* **The corpus view.** How often each entity is present on a page, and which labels are common, across every
  stored page. It is an aggregate query over all `Field` nodes, not a separate tree. It is the memory's model of
  "a page unlike any rule group", the **NEW** class of Part II.

The plot shows where the rule's entity sits across every vendor (grey) against the running example's range: the
corpus holds no useful position, only presence and labels.

In [ ]:
unidentified = groups[groups.vendor == UNIDENTIFIED]
print(f"Unidentified vendor: {len(unidentified)} rule groups from {unidentified.pages.sum()} pages "
      "(every ground-truth invoice in this corpus prints a vendor name; E3 removes them).")
corpus_view = post.corpus_layouts[1]
display(pd.DataFrame({"present on page 1": corpus_view["n_filled"] / corpus_view["docs"],
                      "common labels": [corpus_view["labels"].get(k, []) for k in FEATURES]}, index=FEATURES).round(2))
all_centres = [tuple(r["point"]) for r in graph.cypher(
    "MATCH (p:Page {page_type: 1})-[:`_`*]->(f:Field {key: $key}) WHERE f.point IS NOT NULL RETURN f.point AS point",
    key=EX_KEY)]
ax = viz.page_axes(corpus.pdf(EX_DOC), 1, title=f"'{EX_KEY}': every vendor (grey) vs '{EX_VENDOR}' (blue)", fade=0.6)
ax.scatter(*zip(*all_centres), s=16, color=viz.MUTED, alpha=0.7, zorder=4)
ax.scatter(*zip(*centres), s=26, color=viz.BLUE, edgecolor="white", lw=1, zorder=5)
viz.draw_range(ax, o["range"], viz.BLUE, f"{EX_LAYOUT} range")
plt.show()

---
## Part II: Confidence gating

At test time, the memory sees a page only through an **unguided extraction**: the vendor name it printed, which
fields are filled, and roughly where the boxes are. In the real pipeline this is Gemini's call 0. In this demo it
is a **generated test page**: the held-out ground truth passed through a noise channel tuned to Gemini's measured
errors (box centres off by ~0.057, 5% of fields missed, 2% invented, 10% of names misread). Part III explains
the generator.

In [ ]:
gen = ScenarioGenerator(corpus, graph, cfg)
baseline_cases = gen.baseline()
unseen_cases = [c for c in gen.unseen_and_drift() if c.scenario == "unseen vendor"]

EX_CASE = next(c for c in baseline_cases if c.doc_id == EX_DOC and c.page == 1 and c.scenario == "noisy")
# the unseen-vendor page with the most located fields: the most evidence a stranger's page can offer
UNSEEN_CASE = max(unseen_cases, key=lambda c: (sum(bool(f["bbox"]) for f in c.evidence["fields"].values()), c.case_id))
print(f"Running example (known vendor): {EX_CASE.case_id}, name read as '{EX_CASE.vendor_name}'")
print(f"Contrast (vendor never seen):   {UNSEEN_CASE.case_id}, name read as '{UNSEEN_CASE.vendor_name}'")

### 6. Which rule group is this page? The posterior
During ingestion the matching rule is a hard pass/fail: similar enough, and every shared node in range. A test
page is a noisy extraction, and the hard rule rarely holds for it (E1 measures how rarely). So the gate keeps the rule's
three checks but turns each into a **likelihood**, and asks **how probable** each rule group is, given the evidence
$e$, including the possibility that it is none of them (**NEW**):

$$P(G \mid e) \;\propto\; \underbrace{P(G)}_{\text{prior}}\;\underbrace{P(\hat k \mid G)}_{\text{name}}\;
\underbrace{P(x \mid G)^{1/T}}_{\text{which fields}}\;\underbrace{p(c \mid x, G)}_{\text{where the boxes are}}$$

| ingestion rule | becomes |
|---|---|
| belongs to the vendor? | **name:** a noisy channel. The right key with probability 0.9, a variant spelling or another key otherwise. A **missing** name is no evidence, so every vendor's groups compete on structure |
| similar to the group? | **fields:** the group's nodes with their fill rates, smoothed toward the corpus view, seen through the extraction's miss and false-fill rates. A missing node costs evidence instead of ruling the group out |
| every shared node in range? | **boxes:** each box centre inside a rule's range scores $r/\lvert R\rvert$ and outside it $(1-r)/(1-\lvert R\rvert)$, with $r$ the rule's `reliability`. A stray box costs a bounded amount |
| no group matches → found a new one | **NEW**, with the Good–Turing prior $N_1/N$: groups founded by a single training page ÷ training pages |

Known groups share the rest of the prior by their number of pages. The posterior keeps the ingestion rule's
logic: if the rules never missed, a group with a shared node out of range would get probability 0, just as the range
check rules it out. The fields term does the job of the similarity threshold, but scores the difference instead of
cutting at 0.85.

The bars show each term relative to the best candidate. A long negative bar is the term that ruled a candidate out.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 9))
for ax, c, title in ((axes[0], EX_CASE, "Known vendor"), (axes[1], UNSEEN_CASE, "Vendor never seen")):
    viz.term_bars(post.explain(c.evidence, c.vendor_name, top=5), ax=ax,
                  title=f"{title}: '{c.vendor_name}' (truth: {c.truth})")
plt.tight_layout()
plt.show()
print(f"P(NEW) prior = {post.p_new:.2f}: most rule groups hold a single training page, so 'a page unlike any group' "
      "is the base case.")

### 7. From rule-group belief to a confidence for each rule
The gate shows the rules of the most probable group $G^*$. How far to trust *its* rule for entity $k$? By the law
of total probability over which group the page really belongs to:

$$\text{conf}_k = P(c_k \in R_{G^*,k} \mid e) = \sum_G P(G \mid e)\; P(c_k \in R_{G^*,k} \mid G)$$

| if the page really is from… | chance the value is in $G^*$'s range |
|---|---|
| $G^*$ | its rule's `reliability` |
| another group with a rule for $k$ | how much of its probability mass falls in $R^*$ |
| another group without one, or NEW | the area of $R^*$ (~0.01–0.09) |

So a rule stays trusted when the runner-up group **puts the value in the same place**. It is not just
$P(G^* \mid e)$. It is not just the rule's reliability either, because a reliable rule of the wrong group is still
wrong.

The example is the generated page on which the posterior is **most torn** between a vendor and its structural twin
(two vendors whose invoices look alike, see E2). The two tables break down the confidence of one entity the twins
print in the same place, and one they print in different places.

In [ ]:
twins = find_twins(graph, cfg.twin_min_similarity, cfg.twin_min_shared_rules)
twin_cases = gen.twins(twins)


def ambiguity(c):   # how evenly the posterior splits between the page's vendor and its twin
    vp = gate.page_rules(c.evidence, c.vendor_name)["vendor_posterior"]
    return min(vp.get(c.true_vendor, 0.0), vp.get(c.meta["twin"], 0.0))


TWIN_CASE = max((c for c in twin_cases if not all(c.meta["agree"].values())), key=ambiguity)
TWIN_A, TWIN_B = TWIN_CASE.true_vendor, TWIN_CASE.meta["twin"]
LAYOUT_A, LAYOUT_B = TWIN_CASE.meta["own_layout"], TWIN_CASE.meta["twin_layout"]
print(f"Twins: '{TWIN_A}' and '{TWIN_B}' (structure cosine {TWIN_CASE.meta['twin_similarity']:.2f}). "
      f"Test page of '{TWIN_A}', scenario '{TWIN_CASE.scenario}', name read as '{TWIN_CASE.vendor_name}'.")
g = gate.page_rules(TWIN_CASE.evidence, TWIN_CASE.vendor_name)
print(f"P({TWIN_A}) = {g['vendor_posterior'].get(TWIN_A, 0):.2f}, P({TWIN_B}) = {g['vendor_posterior'].get(TWIN_B, 0):.2f}"
      f" -> MAP rule group {g['chosen']}")
agree = TWIN_CASE.meta["agree"]
for key in [k for k, v in agree.items() if v][:1] + [k for k, v in agree.items() if not v][:1]:
    print(f"\n'{key}' (twins {'agree' if agree[key] else 'disagree'} on where it is):")
    display(gate.confidence_breakdown(TWIN_CASE.evidence, TWIN_CASE.vendor_name, key, top=4).round(3))

In [ ]:
la, lb = post.vendor_layouts[LAYOUT_A], post.vendor_layouts[LAYOUT_B]
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(TWIN_CASE.doc_id), TWIN_CASE.page, ax, fade=0.4,
              title=f"Twin ranges: {TWIN_A} (blue) vs {TWIN_B} (orange)")
for key, same in agree.items():
    viz.draw_range(ax, la["rules"][key]["range"], viz.BLUE, key if not same else None, ls="-")
    viz.draw_range(ax, lb["rules"][key]["range"], viz.ORANGE, None)
plt.show()

### 8. The gate
The confidence becomes a decision:

| confidence | what the model is shown |
|---|---|
| ≥ 0.7 (`gate_box`) | the rule's typical box and its confidence |
| 0.4 – 0.7 (`gate_label`) | label hint only, "position uncertain" |
| < 0.4 | nothing: the rule is **withheld** |

Below is the twin page from section 7: its rules with their decisions, and the same rules drawn on the page, each
range shown as a box (blue), as a label hint only (yellow), or withheld (grey), against the true value boxes.

In [ ]:
g = gate.page_rules(TWIN_CASE.evidence, TWIN_CASE.vendor_name)
display(pd.DataFrame(g["rules"]).rename(columns={"q": "reliability"})
        [["name", "strength", "reliability", "p_filled", "confidence", "gate"]]
        .dropna(subset=["confidence"]).round(3))
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(TWIN_CASE.doc_id), TWIN_CASE.page, ax, fade=0.45,
              title=f"Gate decisions: '{TWIN_A}' page, name not read")
for f in TWIN_CASE.gt["fields"].values():
    if f["bbox"]:
        viz.draw_box(ax, f["bbox"], viz.INK, lw=0.8)
for r in g["rules"]:
    if r["range"] is not None and r["gate"] in ("box", "label", "withheld"):
        viz.draw_range(ax, r["range"], viz.GATE_COLORS[r["gate"]], f"{r['name']} {r['confidence']:.2f}",
                       ls="-" if r["gate"] == "box" else "--")
decisions = {r["gate"] for r in g["rules"] if r["range"] is not None}
ax.legend(handles=[plt.Line2D([], [], color=c, lw=2, ls="-" if k == "box" else "--", label=k)
                   for k, c in viz.GATE_COLORS.items() if k in decisions]
          + [plt.Line2D([], [], color=viz.INK, lw=0.8, label="true value box")], loc="lower left", frameon=True)
plt.show()

---
## Part III: Evaluation 

The gate makes a claim per rule: *"this value is in this range with probability conf"*. That claim can be checked
with ground truth alone: a rule **hits** when the true value centre falls inside its range. So the gate can be
evaluated on **generated test pages**.

**The generator** (`src/scenarios.py`) builds controlled variants of the held-out pages, each with a known right
answer:

| study | generated pages | right answer |
|---|---|---|
| **E1 baseline** | every held-out page, clean, and through the noise channel | its vendor, or NEW if unseen |
| **E2 look-alike vendors** | pages of vendors with a *structural twin*, shown with their own name, no name, or the twin's name | its own vendor |
| **E3 no vendor** | every held-out page with the vendor name removed | its vendor, or NEW |
| **E4 unseen & drift** | unseen vendors; known layouts under a never-seen name; known names on a **redesigned** layout (all boxes moved) | NEW for unseen and redesigned |

Besides calibration, the number that matters most to an LLM is **wrong positions per page**: boxes the gate shows
that point at the wrong spot.

In [ ]:
cases = gen.all_studies()
pages, rules = E.evaluate(gate, cases)
summary = E.study_summary(pages, rules)
shown = E.shown_positions(pages, rules)
display(case_table(cases))
print("What the noise channel did (target: centre miss 0.057, 5% dropped, 2% invented):")
describe_noise(cases)

### E1. Baseline: is the confidence honest?
On ordinary held-out pages: how often does the posterior find the right group, how much of the memory passes the
gate, and do rules with confidence 0.9 hit about 90% of the time? The **reliability curve** should follow the
diagonal. On the **risk–coverage curve**, accuracy should rise as the threshold rises. If it's flat, the
confidence isn't ranking good rules above bad ones.

In [ ]:
display(summary.loc["E1 baseline"])
display(shown.loc["E1 baseline"])
groups = {"E1 noisy pages": rules[(rules.study == "E1 baseline") & (rules.scenario == "noisy")],
          "all generated pages (E1–E4)": rules}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4))
viz.reliability_plot(groups, ax=a1)
viz.coverage_plot(groups, ax=a2, gate_box=cfg.gate_box)
plt.tight_layout(); plt.show()
for name, df in groups.items():
    print(name, {k: round(v, 3) for k, v in E.calibration_scores(df).items()})
display(E.reliability(rules.confidence, rules.hit))

# the hard ingestion rule against the gate, on the same held-out pages
e1 = pd.DataFrame([{"scenario": c.scenario, "ingestion rule picks a group": graph.match_page(c.evidence, c.vendor_name) is not None}
                   for c in cases if c.study == "E1 baseline"])
e1 = e1.groupby("scenario").mean().join(pages[pages.study == "E1 baseline"].groupby("scenario").agg(
    **{"gate picks a group": ("map_is_new", lambda s: 1 - s.mean()), "gate's choice right": ("correct", "mean")}))
print("Held-out pages: how often each rule settles on a rule group")
e1.round(3)

**What to notice.** On noisy held-out pages the gate shows about 4 positions per page and ~95% of them are right,
so the model sees about 0.2 wrong positions per page. Most of the confidence mass sits above 0.9, and there the
hit rate matches (0.947 predicted, 0.960 observed). The middle bins hold few rules and run slightly
overconfident at 0.8–0.9. Noise mostly costs coverage: noisy pages show fewer boxes than clean ones, at the same
precision.

**Why the gate and not the ingestion rule.** Applied to the same held-out pages, the pass/fail rule settles on a
rule group for 41% of clean pages and only 12% of noisy ones: one box nudged out of a range, and the page would
found a new group. The gate settles on a group for 61–69% of pages (two thirds of them are from known vendors),
and its choice is right 89–93% of the time.

### E2. Similar structure, different vendors
Some vendors print near-identical invoices: the same fields, in the same places (structure cosine ≥ 0.9, at least
3 located fields in common). These **twins** are the hardest case for identification, and the clearest case for
why rule confidence is not the same as layout confidence.

Each test page of a twinned vendor is shown three ways: with its own name, with **no name**, and with the **twin's
name** (the extraction misread the vendor).

In [ ]:
display(twins[["vendor_a", "vendor_b", "page", "sim", "shared_rules", "agreeing_rules"]].head(10).round(3))

In [ ]:
twin_cases = [c for c in cases if c.study == "E2 twins"]
twins["disagreeing_rules"] = twins.shared_rules - twins.agreeing_rules
print("Which evidence tells twins apart? (top-1 = the right vendor has the highest posterior)")
display(E.identification(post, twin_cases)["top1_acc"].unstack("evidence"))
display(summary.loc["E2 twins"][["pages", "map_acc", "p_true", "share_box", "hit_box", "brier"]])
print("Rules split by whether the twins' ranges overlap:")
E.twin_agreement(rules)

**What to notice.**
* **Names separate twins and boxes come second.** Without a name, positions (`boxes`) carry most of the
  identification. With the twin's name, the name channel actively misleads and top-1 drops to about a coin flip.
* **Yet box precision barely moves.** When the MAP rule group is the twin's, the rules it shows are mostly still
  right, because the twins print those fields in the same place. The confidence formula credits that agreement:
  rules where twins agree keep high confidence *and* high hit rates.
* **The weak spot:** where twins *disagree*, a misread name still yields ~0.90 confidence against a 0.87 hit
  rate. This is the gate's honest limit. When the evidence confidently points at the wrong twin, the
  total-probability sum has nothing to correct with.

### E3. Documents with no vendor
In this corpus every ground-truth invoice names its vendor. But a missing or illegible name is common in
extraction. E3 removes the name from **every** held-out page, so the posterior has to place the page from its
structure alone. For pages of unseen vendors, the right answer is still NEW.

In [ ]:
no_vendor = [c for c in cases if c.study == "E3 no vendor"]
ident = E.identification(post, no_vendor)
display(ident["top1_acc"].unstack("evidence"))
display(pd.concat({"with name (E1)": summary.loc["E1 baseline"], "no name (E3)": summary.loc["E3 no vendor"]})
        [["pages", "map_acc", "p_new", "share_box", "share_withheld", "hit_box", "brier"]])
display(pd.concat({"with name (E1)": shown.loc["E1 baseline"], "no name (E3)": shown.loc["E3 no vendor"]}))
by_bucket = pages[pages.study == "E3 no vendor"].assign(bucket=lambda d: d.true_vendor.map(corpus.bucket))
by_bucket.groupby(["scenario", "bucket"]).agg(pages=("page", "size"), map_acc=("correct", "mean"),
                                              p_new=("p_new", "mean")).round(3)

**What to notice.**
* Without a name, identification rests on structure, and **positions do most of the work** (boxes alone beat
  fields alone). MAP accuracy falls from ~0.9 to ~0.74.
* The cost is highest for **thin groups** (2–3 invoices in memory). Clean pages of unseen vendors are also more
  often claimed by a known rule group than noisy ones. Exact positions can make a stranger's page resemble a
  known group, while noise breaks the resemblance and sends the page to NEW.
* **The gate absorbs most of it.** Box precision stays ~0.94 and wrong positions per page barely move, because a
  wrong-but-similar rule group usually shares the positions it is confident about.

### E4. Unseen vendors and layout drift
These are the pages where a remembered position is most likely to be wrong.
* **Unseen vendor:** real held-out vendors with no invoices in memory.
* **Known layout, new name:** a known vendor's page under a name the memory has never seen (a rebrand, or a
  badly misread name). Should structure override the name?
* **Known name, redesigned layout:** the vendor's own name, but every box moved down a quarter page. The name
  says "trust me" and the boxes say otherwise.

In [ ]:
display(summary.loc["E4 unseen & drift"][["pages", "map_acc", "p_true", "p_new", "rules", "share_box", "hit_box"]])
display(shown.loc["E4 unseen & drift"])

In [ ]:
redesign = next(c for c in cases if c.scenario == "known name, redesigned layout" and c.doc_id == EX_DOC and c.page == 1)
g_red = gate.page_rules(redesign.evidence, redesign.vendor_name)
remembered = post.vendor_layouts[EX_LAYOUT]["rules"]
fig, ax = plt.subplots(figsize=(7, 9))
viz.page_axes(corpus.pdf(EX_DOC), 1, ax, fade=0.5,
              title=f"Redesign: P(NEW) = {g_red['p_new']:.2f}, "
                    f"{sum(r['gate'] == 'box' for r in g_red['rules'])} boxes shown")
for f in redesign.gt["fields"].values():
    if f["bbox"]:
        viz.draw_box(ax, f["bbox"], viz.BLUE)
for key, r in remembered.items():
    viz.draw_range(ax, r["range"], viz.MUTED, key)
for r in g_red["rules"]:
    if r["gate"] == "box":
        viz.draw_range(ax, r["range"], viz.AQUA, f"shown: {r['name']}", ls="-")
ax.legend(handles=[plt.Line2D([], [], color=viz.BLUE, lw=2, label="true boxes after the redesign"),
                   plt.Line2D([], [], color=viz.MUTED, lw=2, ls="--", label=f"remembered ranges of {EX_LAYOUT}"),
                   plt.Line2D([], [], color=viz.AQUA, lw=2, label="shown by the gate")], loc="lower left", frameon=True)
plt.show()

**What to notice.**
* **Unseen vendors and redesigns go to NEW.** P(NEW) averages 0.96 on unseen vendors and 0.81 on redesigned
  layouts, and the gate shows under 0.2 boxes per page: fewer than 0.1 wrong positions per page. On the redesign
  above, the vendor's name points at its remembered rule group, but every box contradicts it, so nothing is shown.
* **Known layout, new name:** structure partly overrides the unknown name. About half the pages are still placed
  with the right vendor, and the boxes shown there are 96% right.

### E5. Sensitivity: where does it break?
The same studies under heavier box noise, and with the field term **tempered** ($T$ > 1 flattens naive Bayes,
whose independent-field assumption overcounts correlated evidence). Read `gate_box`, `field_temperature` and
`box_range_min` against this table.

In [ ]:
sweep = []
for box_miss in (0.0, 0.03, 0.057, 0.1):
    g_cfg = replace(cfg, noise_box_miss=box_miss)
    sweep_gen = ScenarioGenerator(corpus, graph, g_cfg)
    sweep_cases = [c for c in sweep_gen.baseline() + sweep_gen.no_vendor() + sweep_gen.unseen_and_drift()
                   if c.meta["noisy"]]
    for T in (1.0, 2.0, 4.0):
        p_, r_ = E.evaluate(gate.with_posterior(post.with_settings(field_temperature=T)), sweep_cases)
        shown_ = r_[r_.gate == "box"]
        sweep.append({"box_miss": box_miss, "T": T, **E.calibration_scores(r_), "map_acc": p_.correct.mean(),
                      "share_box": (r_.gate == "box").mean(), "box_precision": shown_.hit.mean(),
                      "wrong_per_page": (~shown_.hit).sum() / len(p_)})
sweep = pd.DataFrame(sweep)
display(sweep.round(3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for T, color in zip((1.0, 2.0, 4.0), (viz.BLUE, viz.ORANGE, viz.AQUA)):
    s = sweep[sweep["T"] == T]
    a1.plot(s.box_miss, s.map_acc, marker="o", color=color, lw=2, label=f"T = {T:g}")
    a2.plot(s.box_miss, s.brier, marker="o", color=color, lw=2, label=f"T = {T:g}")
a1.set(xlabel="mean box-centre miss", ylabel="MAP accuracy", ylim=(0, 1)); a1.set_title("Identification", loc="left")
a2.set(xlabel="mean box-centre miss", ylabel="Brier score (lower is better)"); a2.set_title("Calibration", loc="left")
a1.axvline(0.057, color=viz.AXIS, ls="--", lw=1); a2.axvline(0.057, color=viz.AXIS, ls="--", lw=1)
a1.legend(); plt.tight_layout(); plt.show()

**What to notice.**
* **The gate fails safe.** As box noise doubles past Gemini's level (0.057 → 0.1), identification drops to ~0.55.
  But the gate shows fewer positions, so wrong positions per page *fall* rather than rise. The cost is coverage,
  not trust.
* **Tempering doesn't help here.** $T$ = 2 or 4 leaves calibration flat at realistic noise and worsens it at high
  noise. On this corpus the field term isn't the source of overconfidence. To tolerate noisier extractions, widen
  `box_range_min` instead.

### Overview: every scenario at once

In [ ]:
viz.gate_mix_plot(summary)
plt.show()
shown

---
## Part IV: Does the graph help the LLM?
`gating.ipynb` runs the real comparison with Gemini: call 0 (unguided) and the gated pass (gated memory) on the
held-out third, saved under `results/`. This cell summarizes those runs when they're present.

Those saved passes were guided by `demo.ipynb`'s memory graph. Until the gated pass is re-run with
this memory's guidance, this section measures the old memory, not the rule groups.

In [ ]:
llm = E.llm_fields(corpus)
if llm is None:
    display(Markdown(f"*No saved Gemini passes under `{cfg.results_dir.resolve()}`. Run `gating.ipynb` (or copy its "
                     "`results/` folder here) to fill this section.*"))
else:
    display(llm.groupby("setup").apply(E.summarize, include_groups=False)[E.LLM_METRICS].round(3))
    display(E.llm_table(llm))

In [ ]:
graph.close()